# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL. This dataset presents a tabular compilation of 77 cancer survivors with second primary colorectal cancer, including a range of clinical and pathological variables. The data supports investigation of clinicopathological predictors and MSI-H phenotype distribution.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset name:\n", metadata.name)
print("\nDataset Description:\n", metadata.description)
print("\nFAIR² Dataset identifier:", metadata.identifier)
print("\nPublished date:", metadata.datePublished)


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below, we enumerate all record sets and their contained fields, showing their unique `@id` references for further data extraction steps.

In [ ]:
# List all record sets by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets in the dataset.\n")
for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")

    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    if not fields:
        print("  Fields: None detected.")
    else:
        print("  Fields:")
        for f in fields:
            print(f"    - @id: {f['@id']}, name: {f.get('name', 'N/A')}, dataType: {f.get('dataType', 'N/A')}")
    print("")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis. 
You can select specific record set and field `@id`s identified in the previous step for targeted analysis.

> All entities are referenced by their `@id` as required.

In [ ]:
# Helper function to create a readable name for each DataFrame from the record set @id
def df_name_from_id(record_set_id):
    return record_set_id.split('/')[-1]

# Extract all record sets and their records
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {df.shape[0]} records for record set: {rs_id}")
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")

if dataframes:
    # Pick the first available record set for inspection
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in first record set ({first_rs_id}):\n{dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())
else:
    print("No data extracted from record sets.")

## 4. Exploratory Data Analysis (EDA)
Let's perform common data processing steps, such as filtering records based on specific criteria, normalizing a numeric field, and grouping by a categorical field.

We operate on the primary tabular record set, referencing its and its fields' `@id`s as required.

In [ ]:
# Choose the main tabular record set and its field @ids
# We'll pick the first available record set loaded above, but you can select one by @id
record_set_id = first_rs_id
df = dataframes[record_set_id].copy()

# Print all column names with their @id
print(f"All fields in record set {record_set_id}:")
for col in df.columns:
    print('-', col)

# Choose a numeric field by its @id for analysis (e.g., 'Age' or 'IntervalBetweenDiagnoses'), adjust this as needed
# We'll attempt to pick a suitable field for demonstration
numeric_field_candidates = [col for col in df.columns if 'Age' in col or 'Interval' in col or df[col].dtype in [int, float]]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
else:
    numeric_field_id = df.columns[0]  # fallback

print(f"\nSelected numeric field for EDA: {numeric_field_id}")

# Try thresholding (assuming field is numeric). Adjust threshold as appropriate.
try:
    threshold = df[numeric_field_id].quantile(0.5)  # median as an example threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f} (median):")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
except Exception as e:
    print('Could not filter and normalize field:', numeric_field_id, '\nReason:', str(e))

# Pick a group field, e.g. gender or anatomical location
group_field_candidates = [col for col in df.columns if 'Sex' in col or 'Gender' in col or 'Anatomical' in col or 'Subtype' in col or df[col].dtype == object]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
    print(f"\nSelected group field for EDA: {group_field_id}")
    if group_field_id in filtered_df.columns:
        # Use mean aggregation for numeric
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id} for filtered data:")
        display(grouped_df)
else:
    print("No obvious group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field distribution
if 'filtered_df' in locals() and not filtered_df.empty and numeric_field_id in filtered_df:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=10, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# If a group field for box plot
if 'group_field_id' in locals() and group_field_id in filtered_df:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id], palette='pastel')
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, you have:
- Loaded the FAIR² dataset metadata and records using their Croissant schema via the `mlcroissant` library.
- Explored available record sets, fields, and referenced entities by their `@id`.
- Extracted tabular data into pandas DataFrames, performed filtering, normalization, and grouping operations using referenced field `@id`s.
- Visualized core distributions and group-wise comparisons in the dataset.

This notebook provides a reproducible and extensible foundation for clinicopathological data analysis using the FAIR² dataset and Croissant standard metadata.